In [0]:
# Create SCD Type 2 table
spark.sql("""
CREATE TABLE IF NOT EXISTS scd2 (
    customer_id STRING,
    name STRING,
    address STRING,
    effective_date DATE,
    end_date DATE,
    current_flag BOOLEAN
)
USING DELTA
""")

# Example SCD Type 2 insert values
from pyspark.sql.functions import lit, current_date

# New data to insert
new_data = [
    ("C001", "Alice", "123 Main St"),
    ("C002", "Bob", "456 Oak Ave")
]
columns = ["customer_id", "name", "address"]
ndf = spark.createDataFrame(new_data, columns)

# Add SCD2 columns
scd2_df = ndf.withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("current_flag", lit(True))

# Insert into SCD2 table
scd2_df.write.format("delta").mode("append").saveAsTable("scd2")

display(spark.table("scd2"))

In [0]:
# Create a source DataFrame for SCD Type 2 processing
data = [
    ("C001", "Alice", "H"),
    ("C002", "Bob", "y"),
    ("C003", "Charlie", "m"),
    ("c004","sanath","banglore")
]
source_columns = ["customer_id", "name", "address"]
data1 = spark.createDataFrame(data, source_columns)

data1.createOrReplaceTempView("data2")
display(data1)

In [0]:
%sql
insert into scd2  (
select s.customer_id,s.name,s.address,current_date() as effective_date,null as end_date,True as current_flag from data2 s
left join scd2 t
on s.customer_id=t.customer_id where  s.address<>t.address)

In [0]:
%sql
select * from scd2;